# On-policy distillation from first principles

This notebook implements the core idea in Kevin Lu and Thinking Machines Lab's [On-Policy Distillation](https://thinkingmachines.ai/blog/on-policy-distillation/) article (October 27, 2025; DOI [10.64434/tml.20251026](https://doi.org/10.64434/tml.20251026)). The corresponding [Tinker recipe](https://tinker-docs.thinkingmachines.ai/cookbook/recipes/distillation/) is maintained separately and may change models or hyperparameters.

The experiment is CPU-only PyTorch. It trains tiny autoregressive students so that the algorithm and state-distribution issue remain visible. It does **not** claim to reproduce the article's reasoning benchmarks or infrastructure.

## Learning objectives

1. Distinguish SFT/offline distillation, outcome-reward RL, and on-policy distillation (OPD).
2. Implement reverse KL from a student to a fixed teacher on student-generated prefixes.
3. Isolate why the prefix distribution matters by comparing against dense distillation on fixed teacher prefixes.
4. Explain support, conditioning, rollout freshness, token masking, compute, and memory requirements.
5. State what OPD can copy efficiently and what still requires search, RL, data, or a stronger initialization.

## 1. Put OPD between SFT and RL

| Method | Prefix/state source | Feedback | Main limitation |
|---|---|---|---|
| SFT or offline distillation | Fixed dataset, often teacher-generated | Dense next-token target or teacher distribution | Does not directly supervise states caused by the student's own mistakes |
| Outcome-reward RL | Current student | Usually one sparse scalar per completion | Credit assignment and high-variance sampling |
| On-policy distillation | Current student | Dense teacher distribution at each eligible token | Requires teacher inference and useful teacher behavior within student support |

OPD combines the state coverage of on-policy sampling with dense supervision from a teacher. It is closely related to online imitation learning and DAgger: train on states induced by the learner, not only states induced by the expert.

## 2. Algorithm

Let `student` be the trainable policy, `teacher` a fixed policy, and `D` a prompt distribution. One update is:

```text
prompts <- sample(D)
trajectories <- sample_autoregressively(student, prompts)
for every student-token position t:
    s_t <- prompt plus the student's preceding tokens
    p_t <- student distribution at s_t
    q_t <- teacher distribution at exactly the same s_t
    loss_t <- KL(p_t || q_t)
loss <- masked mean of loss_t
update student; keep teacher fixed
```

Prompt, padding, and environment/tool tokens remain in the context but normally do not receive policy loss. Multi-turn training should apply the mask only to tokens actually generated by the student. Fresh rollouts keep the training states close to the current policy; excessive rollout reuse gradually makes the update off-policy.

## 3. Reverse KL at a student-visited state

For a prefix $s_t$ visited by the student, the token loss is

$$D_{\mathrm{KL}}(\pi_S(\cdot\mid s_t)\,\|\,\pi_T(\cdot\mid s_t)) = \sum_a \pi_S(a\mid s_t) [\log \pi_S(a\mid s_t)-\log \pi_T(a\mid s_t)].$$

Reverse KL penalizes probability the student assigns where the teacher assigns little probability. It is mode-seeking when the teacher is multimodal, so diversity should be measured rather than assumed.

With full logits, the small-vocabulary implementation below sums over all actions exactly. Real large-vocabulary systems can use specialized/chunked kernels or sampled estimators. Prefix sampling is performed without gradient; each update differentiates the KL on the sampled states, then refreshes those states after the policy changes. The article reports using zero future discount: the teacher's signal at a token trains that token directly rather than being propagated as a future return.

In [ ]:
import copy
import platform

import torch
from torch import nn

print(f"Python: {platform.python_version()}")
print(f"PyTorch: {torch.__version__}")
print(f"Device: cpu")

## 4. A controlled recovery task

There are five prompts, each naming a target token `0` through `4`. A good response emits the target and then `EOS`. The fixed teacher behaves as follows on **any** prefix:

- before the target appears, put 96% probability on the target;
- after the target or an early `EOS`, put 96% probability on `EOS`;
- leave 4% probability across the other actions so every KL is finite.

Clean teacher trajectories contain only `[target, EOS, EOS]`. They never contain a wrong first token. Student rollouts initially do, so OPD can ask the teacher what to do after those errors. This creates a measurable covariate-shift test without pretending that the toy task is language reasoning.

In [ ]:
SEED = 7
torch.manual_seed(SEED)
torch.set_num_threads(1)

N_TARGETS = 5
EOS = 5
START = 6
N_ACTIONS = 6
HIDDEN_SIZE = 32
RESPONSE_LENGTH = 3
BATCH_REPEATS = 16
UPDATES = 100
LEARNING_RATE = 2e-2

ACTION_NAMES = ["0", "1", "2", "3", "4", "EOS"]

## 5. Tiny autoregressive student

The prompt initializes a GRU state. At each position, the previous response action enters the GRU and the head predicts the next action. `sample` defines the student's on-policy state distribution; `logits_on_prefixes` replays chosen actions so gradients can be computed at each visited state.

In [ ]:
class StudentPolicy(nn.Module):
    def __init__(self):
        super().__init__()
        self.prompt_embedding = nn.Embedding(N_TARGETS, HIDDEN_SIZE)
        self.action_embedding = nn.Embedding(N_ACTIONS + 1, HIDDEN_SIZE)
        self.cell = nn.GRUCell(HIDDEN_SIZE, HIDDEN_SIZE)
        self.head = nn.Linear(HIDDEN_SIZE, N_ACTIONS)

    def logits_on_prefixes(self, prompts, actions):
        hidden = self.prompt_embedding(prompts)
        previous = torch.full_like(prompts, START)
        logits = []
        for position in range(actions.shape[1]):
            hidden = self.cell(self.action_embedding(previous), hidden)
            logits.append(self.head(hidden))
            previous = actions[:, position]
        return torch.stack(logits, dim=1)

    @torch.no_grad()
    def generate(self, prompts, sample=True, temperature=1.0):
        hidden = self.prompt_embedding(prompts)
        previous = torch.full_like(prompts, START)
        actions = []
        for _ in range(RESPONSE_LENGTH):
            hidden = self.cell(self.action_embedding(previous), hidden)
            logits = self.head(hidden) / temperature
            if sample:
                previous = torch.multinomial(logits.softmax(dim=-1), 1).squeeze(1)
            else:
                previous = logits.argmax(dim=-1)
            actions.append(previous)
        return torch.stack(actions, dim=1)

## 6. Fixed teacher and exact KL

The teacher is analytic so its behavior is inspectable. In a real run, `teacher_log_probs` would be a frozen language-model forward pass on the same prompt and response prefix. No sampled teacher token is needed when full teacher logits are available.

In [ ]:
def teacher_log_probs(prompts, actions, smoothing=0.04):
    batch_size = prompts.shape[0]
    finished = torch.zeros(batch_size, dtype=torch.bool)
    log_probs = []

    for position in range(actions.shape[1]):
        wanted = torch.where(finished, torch.full_like(prompts, EOS), prompts)
        probs = torch.full(
            (batch_size, N_ACTIONS),
            smoothing / (N_ACTIONS - 1),
        )
        probs.scatter_(1, wanted[:, None], 1.0 - smoothing)
        log_probs.append(probs.log())
        finished |= actions[:, position].eq(prompts) | actions[:, position].eq(EOS)

    return torch.stack(log_probs, dim=1)


def reverse_kl_loss(student, prompts, actions):
    student_log_probs = student.logits_on_prefixes(prompts, actions).log_softmax(dim=-1)
    fixed_teacher_log_probs = teacher_log_probs(prompts, actions)
    student_probs = student_log_probs.exp()
    token_kl = (
        student_probs * (student_log_probs - fixed_teacher_log_probs)
    ).sum(dim=-1)
    return token_kl.mean(), token_kl.detach()

## 7. Matched comparison: change only the prefixes

Both students start with identical weights, see every prompt equally often, receive the full teacher distribution, take 100 optimizer steps, and use the same optimizer.

- **Offline:** compute dense reverse KL only on fixed clean teacher trajectories.
- **On-policy:** sample fresh actions from the current student, then compute the same dense reverse KL on those prefixes.

This offline condition is stronger than hard-label SFT because it also receives the teacher's non-target probabilities. The experiment therefore isolates state coverage rather than soft versus hard labels.

In [ ]:
base_student = StudentPolicy()
offline_student = copy.deepcopy(base_student)
on_policy_student = copy.deepcopy(base_student)

offline_optimizer = torch.optim.Adam(offline_student.parameters(), lr=LEARNING_RATE)
on_policy_optimizer = torch.optim.Adam(on_policy_student.parameters(), lr=LEARNING_RATE)
history = {"offline": [], "on_policy": []}

for update in range(UPDATES):
    prompts = torch.arange(N_TARGETS).repeat_interleave(BATCH_REPEATS)

    teacher_actions = torch.stack(
        [prompts, torch.full_like(prompts, EOS), torch.full_like(prompts, EOS)],
        dim=1,
    )
    offline_loss, _ = reverse_kl_loss(offline_student, prompts, teacher_actions)
    offline_optimizer.zero_grad()
    offline_loss.backward()
    offline_optimizer.step()

    student_actions = on_policy_student.generate(prompts, sample=True)
    on_policy_loss, _ = reverse_kl_loss(on_policy_student, prompts, student_actions)
    on_policy_optimizer.zero_grad()
    on_policy_loss.backward()
    on_policy_optimizer.step()

    history["offline"].append(offline_loss.detach().item())
    history["on_policy"].append(on_policy_loss.detach().item())

    if update in {0, 9, 49, 99}:
        print(
            f"update {update + 1:3d} | "
            f"offline KL {offline_loss.detach().item():.4f} | "
            f"on-policy KL {on_policy_loss.detach().item():.4f}"
        )

## 8. Evaluate ordinary and recovery states

The ordinary test starts cleanly and greedily generates three actions. The recovery test intervenes after the first state and forces the wrong token `(target + 1) mod 5`. It then checks whether the student's next action recovers by emitting the target. Fixed teacher trajectories never contain this forced-error prefix.

In [ ]:
@torch.no_grad()
def recovery_actions(student):
    prompts = torch.arange(N_TARGETS)
    forced_wrong = (prompts + 1) % N_TARGETS
    hidden = student.prompt_embedding(prompts)
    start = torch.full_like(prompts, START)
    hidden = student.cell(student.action_embedding(start), hidden)
    hidden = student.cell(student.action_embedding(forced_wrong), hidden)
    return student.head(hidden).argmax(dim=-1)


def format_actions(actions):
    return [" ".join(ACTION_NAMES[token] for token in row) for row in actions.tolist()]


prompts = torch.arange(N_TARGETS)
for name, student in [
    ("offline prefixes", offline_student),
    ("on-policy prefixes", on_policy_student),
]:
    ordinary = student.generate(prompts, sample=False)
    recovered = recovery_actions(student)
    ordinary_success = ordinary[:, 0].eq(prompts).float().mean().item()
    recovery_success = recovered.eq(prompts).float().mean().item()
    print(f"{name}:")
    print(f"  greedy responses: {format_actions(ordinary)}")
    print(f"  ordinary success: {ordinary_success:.0%}")
    print(f"  action after forced error: {[ACTION_NAMES[x] for x in recovered.tolist()]}")
    print(f"  recovery success: {recovery_success:.0%}\n")

assert on_policy_student.generate(prompts, sample=False)[:, 0].eq(prompts).all()
assert recovery_actions(on_policy_student).eq(prompts).all()

## 9. Interpret the result carefully

Both students learn the clean trajectory. The offline student learns `EOS` after every first token it observed in training, so it also emits `EOS` after a forced wrong token. The on-policy student encountered wrong prefixes during early sampling and learned the teacher's recovery action there.

This demonstrates one mechanism, not a universal performance theorem. If the deployment states are fully covered by offline data, offline distillation can be enough. If the student never samples a crucial teacher strategy, OPD cannot learn it merely by using dense KL. Conversely, a teacher may be unreliable on malformed prefixes, so broader coverage can expose bad teacher behavior as well as useful recovery behavior.

## 10. Conditioning, system prompts, and solidification

A system prompt is part of the policy state. If the same system prompt appears in every training example, it becomes statistically constant; the model can absorb the associated behavior without learning that the prompt is a controllable switch. OPD does not automatically prevent that.

Design the conditioning distribution around the behavior required at deployment:

- vary system instructions when behavior should remain instruction-sensitive;
- include absent, alternative, and conflicting conditions only when those cases have a defined desired behavior;
- evaluate counterfactual prompt changes rather than only the training template;
- keep tool/environment tokens in context but mask them from the student loss;
- mix domains or use regularization/replay if narrow distillation causes regressions elsewhere.

A later large 'hero run' cannot guarantee recovery of an invariance that the data never identifies. Prompt control, data variation, and held-out evaluations should be designed before the expensive run.

## 11. Compute, memory, and storage model

Use distinct symbols: $P_S$ and $P_T$ are student and teacher parameters, $T$ is the number of processed tokens, $B$ is batch size, $L$ is sequence length, $V$ is vocabulary size, and $D$ is hidden width. For a dense Transformer, useful first-order estimates are:

| Item | Approximate cost | Notes |
|---|---:|---|
| Student generation forward | $2P_ST$ FLOPs | KV caching changes token-by-token constants |
| Student training forward + backward | $6P_ST$ FLOPs | about $2P_ST$ forward plus $4P_ST$ backward |
| Frozen teacher forward | $2P_TT$ FLOPs | no teacher backward or optimizer |
| Full OPD update, no forward reuse | $8P_ST + 2P_TT$ FLOPs | rough lower-order-free accounting |
| FP32 Adam student state | about $16P_S$ bytes | weights, gradients, first and second moments |
| Frozen teacher weights | dtype bytes $\times P_T$ | e.g. roughly $2P_T$ bytes in BF16 |
| Full logits | $O(BLV)$ values per model | often chunk or fuse KL instead of retaining both |
| Activations | roughly $O(BLD \times \text{layers})$ | checkpointing trades extra compute for memory |
| Rollout storage | $O(BL)$ token IDs plus metadata | log probabilities/logits greatly increase it |

The familiar `$2PT$ forward, $4PT$ backward` rule describes parameter-multiply FLOPs, not bytes. Attention can be important at long context, and real totals also include sampling, communication, padding, recomputation, and optimizer work. LoRA reduces gradients and optimizer state for trainable parameters, but both base-model forwards still run. Asynchronous OPD adds queues and policy-version/staleness bookkeeping.

## 12. Relate the toy loop to the reported work

The 2025 article reports a Qwen3 reasoning experiment reaching about 70% AIME accuracy after roughly 150 updates, 77,000 prompts, and four samples per prompt. In a matched teacher-recovery study it reports 7--10 times fewer gradient steps and 50--100 times less cumulative compute than RL. The maintained recipe now uses Qwen3.5-9B-Base/Qwen3.5-9B and documents multi-turn masking, multiple teachers, LoRA settings, and current reproduction commands.

Do not transfer those numbers to this notebook. The toy loop has an analytic teacher, exact six-action KL, short synchronous trajectories, no distributed inference, and no natural-language evaluation. The source's broader claim is that dense teacher information can make imitation much more sample-efficient than discovering the same behavior from sparse rewards. RL or search can still be needed to discover strategies that the available teacher does not already express.

## 13. Exercises and qualification-exam questions

1. Replace reverse KL with forward KL $D_{KL}(teacher || student)$. Compare recovery and entropy, and explain the different mode-covering behavior.
2. Reuse each student rollout for 1, 5, and 20 updates. Quantify policy staleness and decide when the procedure is no longer meaningfully on-policy.
3. Reduce sampling temperature until the initial student almost never visits some errors. Relate the result to the support condition.
4. Make the teacher wrong on one recovery prefix. Explain why OPD copies that failure efficiently.
5. Add a scalar success reward and implement REINFORCE on the same task. Compare trajectories and gradient steps under equal environment samples.
6. Add a system-condition token that reverses the target mapping. Train with one condition only, then both; test whether the condition remains controllable.
7. Replace the analytic teacher with a separately trained frozen network and inspect its calibration off the clean trajectory.
8. Mask one response position from KL. Connect the result to masking environment/tool tokens in multi-turn agent training.
9. Estimate GPU memory for a chosen student/teacher pair under full fine-tuning and LoRA, stating every dtype and activation assumption.
10. Design an experiment that compares OPD, SFT, and RL under equal wall-clock, total FLOPs, teacher calls, and generated-token budgets. Explain why no single accounting is neutral.